In [1]:
# ==== Edit this first ====
REPO_URL = "https://github.com/anshularavind/cs188-default-project.git"
REPO_DIR = "/content/cs188-default-project"

# ==== Training options ====
USE_15M_DIFFUSION = True
EPOCHS = 40
BATCH_SIZE = 64
MAX_EPISODES = None  # e.g. 300 for a faster run

import os
import subprocess
import yaml

def sh(cmd):
    print("+", cmd)
    subprocess.check_call(cmd, shell=True)

# Clone (or refresh) repo
if not os.path.exists(REPO_DIR):
    sh(f"git clone {REPO_URL} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull --ff-only")

REPO_ROOT = REPO_DIR
PROJECT_DIR = os.path.join(REPO_ROOT, "cabinet_door_project")
ROBOCASA_DIR = os.path.join(REPO_ROOT, "robocasa")

# 1) Core install (apt + pip + editable installs)
os.chdir(REPO_ROOT)
sh(f"python {PROJECT_DIR}/99_colab_setup.py")

# 2) Kitchen assets + macros (robocasa is outside cabinet_door_project)
# Asset download is large (~10GB) and can take a while.
sh(f"bash -lc \"printf 'y\\n' | python {ROBOCASA_DIR}/robocasa/scripts/download_kitchen_assets.py\"")
sh(f"python {ROBOCASA_DIR}/robocasa/scripts/setup_macros.py")

# Verify after assets/macros are present
sh(f"python {PROJECT_DIR}/00_verify_installation.py")

# 3) Dataset + augmentation
sh(f"python {PROJECT_DIR}/04_download_dataset.py")
os.chdir(PROJECT_DIR)
sh("python 05b_augment_handle_data.py")

# 4) Optional ~15M diffusion config
config_path = "configs/diffusion_policy.yaml"
if USE_15M_DIFFUSION:
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    cfg["base_channels"] = 112
    cfg["channel_mults"] = [1, 2, 4]
    cfg["num_res_blocks"] = 2
    cfg["cond_dim"] = 448
    cfg["num_diffusion_iters"] = 100
    cfg["num_inference_iters"] = 16
    config_path = "configs/diffusion_policy_15m_colab.yaml"
    with open(config_path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print("Using 15M diffusion config:", config_path)

# 5) Train only
train_cmd = f"python 06_train_policy.py --config {config_path} --epochs {EPOCHS} --batch_size {BATCH_SIZE}"
if MAX_EPISODES is not None:
    train_cmd += f" --max_episodes {int(MAX_EPISODES)}"
sh(train_cmd)

print("Training finished. Checkpoints are in /tmp/cabinet_policy_checkpoints")


+ git -C /content/cs188-default-project pull --ff-only
+ python /content/cs188-default-project/cabinet_door_project/99_colab_setup.py
+ bash -lc "printf 'y\n' | python /content/cs188-default-project/robocasa/robocasa/scripts/download_kitchen_assets.py"
+ python /content/cs188-default-project/robocasa/robocasa/scripts/setup_macros.py


CalledProcessError: Command 'python /content/cs188-default-project/robocasa/robocasa/scripts/setup_macros.py' returned non-zero exit status 1.

In [2]:
import os, glob
from pathlib import Path

candidates = [
    "/tmp/cabinet_policy_checkpoints",
    "/tmp/cabinet_stage_bc_checkpoints",
]

found = False
for d in candidates:
    print(f"\nChecking: {d}")
    if not os.path.exists(d):
        print("  (missing)")
        continue
    files = sorted(glob.glob(os.path.join(d, "*.pt")))
    if not files:
        print("  no .pt files yet")
        continue
    found = True
    for f in files:
        p = Path(f)
        print(f"  {p.name}  size={p.stat().st_size/1e6:.2f}MB  path={f}")

if not found:
    print("\nNo checkpoint .pt files found in /tmp yet.")
    print("If training is still running, a file appears after the first saved epoch.")



Checking: /tmp/cabinet_policy_checkpoints
  best_policy.pt  size=185.37MB  path=/tmp/cabinet_policy_checkpoints/best_policy.pt
  final_policy.pt  size=185.37MB  path=/tmp/cabinet_policy_checkpoints/final_policy.pt

Checking: /tmp/cabinet_stage_bc_checkpoints
  (missing)


In [3]:
# Drop-in cell: auto-backup /tmp checkpoints to Google Drive (during + after training)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time, datetime, threading
from pathlib import Path

TMP_DIR = Path("/tmp/cabinet_policy_checkpoints")          # diffusion default
# TMP_DIR = Path("/tmp/cabinet_stage_bc_checkpoints")      # staged BC default (if needed)

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_DIR = Path(f"/content/drive/MyDrive/cabinet_door_backups/{RUN_ID}")
DRIVE_CKPT_DIR = DRIVE_DIR / "checkpoints"
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("Watching:", TMP_DIR)
print("Backing up to:", DRIVE_CKPT_DIR)

def sync_once():
    if not TMP_DIR.exists():
        return
    for name in ["best_policy.pt", "final_policy.pt"]:
        src = TMP_DIR / name
        if src.exists():
            dst = DRIVE_CKPT_DIR / name
            shutil.copy2(src, dst)
    # Optional: copy all files in checkpoint dir
    for f in TMP_DIR.glob("*"):
        if f.is_file():
            shutil.copy2(f, DRIVE_CKPT_DIR / f.name)

# Background autosync every 60s
stop_flag = {"stop": False}
def loop():
    while not stop_flag["stop"]:
        try:
            sync_once()
        except Exception as e:
            print("sync warning:", e)
        time.sleep(60)

t = threading.Thread(target=loop, daemon=True)
t.start()

print("Autosync started. It will keep copying checkpoints to Drive every 60s.")
print("When training ends, run this in a new cell to finalize:")
print("stop_flag['stop']=True; sync_once(); print('final sync complete')")


Mounted at /content/drive
Watching: /tmp/cabinet_policy_checkpoints
Backing up to: /content/drive/MyDrive/cabinet_door_backups/20260319_205601/checkpoints
Autosync started. It will keep copying checkpoints to Drive every 60s.
When training ends, run this in a new cell to finalize:
stop_flag['stop']=True; sync_once(); print('final sync complete')
